# Inteligencia de Negocios - Análisis comercial.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sqlalchemy import create_engine, text
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')


Establecemos ruta a carpeta:

In [3]:
path = 'C:\\Users\\luis\\Dropbox\\Data-science\\Proyectos\\Analisis-Comercial\\Dataset2\\'

Cargamos las tablas necesarias para respectivo análisis exploratorio de datos.

In [4]:
orders = pd.read_csv(path + 'olist_orders_dataset.csv', sep=',')
order_items = pd.read_csv(path + 'olist_order_items_dataset.csv', sep=',')
payments = pd.read_csv(path + 'olist_order_payments_dataset.csv', sep=',')
reviews = pd.read_csv(path + 'olist_order_reviews_dataset.csv', sep=',', encoding='utf-8')
products = pd.read_csv(path + 'olist_products_dataset.csv', sep=',')
customers = pd.read_csv(path + 'olist_customers_dataset.csv', sep=',')
sellers = pd.read_csv(path + 'olist_sellers_dataset.csv', sep=',')

In [57]:
translate = pd.read_csv(path+'product_category_name_translation.csv', sep=',')
translate.head(10)

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
5,esporte_lazer,sports_leisure
6,perfumaria,perfumery
7,utilidades_domesticas,housewares
8,telefonia,telephony
9,relogios_presentes,watches_gifts


# 1- Análisis exploratorio de datos.

En primer lugar realizamos un análisis exploratorio de datos, para tener una idea a priori de nuestros conjunto de datos. El objetivo es identificar valores nulos y posibles errores que contengan nuestraas bases de datos, para posteriormente realizar respectiva tratamiento y exportación de bases de datos.

Ante posible presencia de valores nulos, realizamos análisis tabla a tabla y hacemos tratamiento de valores dependiendo el caso. 


## 1.2 EDA para dataset: ordenes.

In [13]:
orders.head(2)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00


In [5]:
orders = orders.rename(columns={
    'order_id': 'id_orden',
    'customer_id': 'id_cliente',
    'order_status': 'estado_orden',
    'order_purchase_timestamp': 'fecha_compra',
    'order_approved_at': 'Hora_orden_aprovada',
    'order_delivered_carrier_date': 'fecha_entrega_transportista',
    'order_delivered_customer_date': 'fecha_entregada_cliente',
    'order_estimated_delivery_date': 'fecha_estimada_entrega'
})

Ahora, procedemos convirtiendo las variables que corresponden al tipo fecha.

In [6]:
orders['fecha_compra'] = pd.to_datetime(orders['fecha_compra'], errors='coerce')
orders['Hora_orden_aprovada'] = pd.to_datetime(orders['Hora_orden_aprovada'], errors='coerce')
orders['fecha_entrega_transportista'] = pd.to_datetime(orders['fecha_entrega_transportista'], errors='coerce')
orders['fecha_entregada_cliente'] = pd.to_datetime(orders['fecha_entregada_cliente'], errors='coerce')
orders['fecha_estimada_entrega'] = pd.to_datetime(orders['fecha_estimada_entrega'], errors='coerce')

In [16]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   id_orden                     99441 non-null  object        
 1   id_cliente                   99441 non-null  object        
 2   estado_orden                 99441 non-null  object        
 3   fecha_compra                 99441 non-null  datetime64[ns]
 4   Hora_orden_aprovada          99281 non-null  datetime64[ns]
 5   fecha_entrega_transportista  97658 non-null  datetime64[ns]
 6   fecha_entregada_cliente      96476 non-null  datetime64[ns]
 7   fecha_estimada_entrega       99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


In [17]:
orders.isnull().sum()

id_orden                          0
id_cliente                        0
estado_orden                      0
fecha_compra                      0
Hora_orden_aprovada             160
fecha_entrega_transportista    1783
fecha_entregada_cliente        2965
fecha_estimada_entrega            0
dtype: int64

## 1.3 EDa para dataset: order_items. 

In [7]:
order_items.head(2)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93


In [8]:
order_items = order_items.rename(columns={
    'order_id': 'id_orden',
    'order_item_id': 'id_pedido_articulo',
    'product_id': 'id_producto',
    'seller_id': 'id_vendedor',
    'shipping_limit_date': 'fecha_limite_envio',
    'price': 'precio',
    'freiht_value': 'valor_flete'    
})

Convertimos variables correspondientes a tipo fecha:

In [9]:
order_items['fecha_limite_envio'] = pd.to_datetime(order_items['fecha_limite_envio'], errors='coerce')

In [22]:
order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   id_orden            112650 non-null  object        
 1   id_pedido_articulo  112650 non-null  int64         
 2   id_producto         112650 non-null  object        
 3   id_vendedor         112650 non-null  object        
 4   fecha_limite_envio  112650 non-null  datetime64[ns]
 5   precio              112650 non-null  float64       
 6   freight_value       112650 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(3)
memory usage: 6.0+ MB


In [23]:
order_items.isnull().sum()

id_orden              0
id_pedido_articulo    0
id_producto           0
id_vendedor           0
fecha_limite_envio    0
precio                0
freight_value         0
dtype: int64

## 1.4 EDA para dataset: payments.

In [24]:
payments.head(2)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39


In [10]:
payments = payments.rename(columns={
    'order_id': 'order_id',
    'payment_sequential': 'secuencia_pago',
    'payment_type': 'tipo_pago',
    'payment_installments': 'plazo_pago',
    'payment_value': 'valor_pago'
})

In [26]:
payments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   order_id        103886 non-null  object 
 1   secuencia_pago  103886 non-null  int64  
 2   tipo_pago       103886 non-null  object 
 3   plazo_pago      103886 non-null  int64  
 4   valor_pago      103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB


In [27]:
payments.isnull().sum()

order_id          0
secuencia_pago    0
tipo_pago         0
plazo_pago        0
valor_pago        0
dtype: int64

## 1.5 EDA para dataset: reviews.

In [11]:
reviews.head(3)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24


In [12]:
reviews = reviews.rename(columns={
    'review_id': 'id_reseña',
    'order_id': 'order_id',
    'review_score': 'puntuación_reseña',
    'review_comment_title': 'titulo_comentario_reseña',
    'review_comment_message': 'Mensaje_comentario_reseña',
    'review_creation_date': 'Fecha_creacion_reseña',
    'review_answer_timestamp': 'Tiempo_estimado_respuesta'
})

Convertimos variables correspondiente al tipo fecha (datetime):

In [13]:
reviews['Fecha_creacion_reseña'] = pd.to_datetime(reviews['Fecha_creacion_reseña'], errors='coerce')
reviews['Tiempo_estimado_respuesta'] = pd.to_datetime(reviews['Tiempo_estimado_respuesta'], errors='coerce')

In [32]:
reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   id_reseña                  99224 non-null  object        
 1   order_id                   99224 non-null  object        
 2   puntuación_reseña          99224 non-null  int64         
 3   titulo_comentario_reseña   11568 non-null  object        
 4   Mensaje_comentario_reseña  40977 non-null  object        
 5   Fecha_creacion_reseña      99224 non-null  datetime64[ns]
 6   Tiempo_estimado_respuesta  99224 non-null  datetime64[ns]
dtypes: datetime64[ns](2), int64(1), object(4)
memory usage: 5.3+ MB


In [33]:
reviews.isnull().sum()

id_reseña                        0
order_id                         0
puntuación_reseña                0
titulo_comentario_reseña     87656
Mensaje_comentario_reseña    58247
Fecha_creacion_reseña            0
Tiempo_estimado_respuesta        0
dtype: int64

In [14]:
reviews['titulo_comentario_reseña'] = reviews['titulo_comentario_reseña'].fillna('Sin titulo')
reviews['Mensaje_comentario_reseña'] = reviews['Mensaje_comentario_reseña'].fillna('Sin comentario')
reviews['Mensaje_comentario_reseña'] = reviews['Mensaje_comentario_reseña'].str.replace(',', ';')

In [15]:
reviews.head(3)

,id_reseña,order_id,puntuación_reseña,titulo_comentario_reseña,Mensaje_comentario_reseña,Fecha_creacion_reseña,Tiempo_estimado_respuesta
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,Sin titulo,Sin comentario,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,Sin titulo,Sin comentario,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,Sin titulo,Sin comentario,2018-02-17,2018-02-18 14:36:24


In [11]:
reviews.isnull().sum()

id_reseña                    0
order_id                     0
puntuación_reseña            0
titulo_comentario_reseña     0
Mensaje_comentario_reseña    0
Fecha_creacion_reseña        0
Tiempo_estimado_respuesta    0
dtype: int64

## 1.6 EDA para dataset: products.

In [34]:
products.head(2)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0


In [16]:
products = products.rename(columns={
    'product_id': 'id_producto',
    'product_category_name': 'nombre_categoria_producto',
    'product_name_lenght': 'nombre_tamaño_producto',
    'product_description_lenght': 'tamaño_descripcion_producto',
    'product_photos_qty': 'cantidad_fotos_producto',
    'product_weight_g': 'peso_producto_gr',
    'product_length_cm': 'tamaño_producto_cm',
    'product_height_cm': 'altura_producto_cm',
    'product_width_cm': 'ancho_producto_cm'
})

In [36]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   id_producto                  32951 non-null  object 
 1   nombre_categoria_producto    32341 non-null  object 
 2   nombre_tamaño_producto       32341 non-null  float64
 3   tamaño_descripcion_producto  32341 non-null  float64
 4   cantidad_fotos_producto      32341 non-null  float64
 5   peso_producto_gr             32949 non-null  float64
 6   tamaño_producto_cm           32949 non-null  float64
 7   altura_producto_cm           32949 non-null  float64
 8   ancho_producto_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


In [37]:
products.isnull().sum()

id_producto                      0
nombre_categoria_producto      610
nombre_tamaño_producto         610
tamaño_descripcion_producto    610
cantidad_fotos_producto        610
peso_producto_gr                 2
tamaño_producto_cm               2
altura_producto_cm               2
ancho_producto_cm                2
dtype: int64

## 1.7 EDA para dataset: customer.

In [39]:
customers.head(2)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP


In [17]:
customers = customers.rename(columns={
    'customer_id': 'id_cliente',
    'customer_unique_id': 'identificador_unico_cliente',
    'customer_zip_code_prefix': 'codigo_postal_cliente',
    'cutomer_city': 'ciudad_cliente',
    'customer_state': 'Estado_cliente'
})

In [42]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   id_cliente                   99441 non-null  object
 1   identificador_unico_cliente  99441 non-null  object
 2   codigo_postal_cliente        99441 non-null  int64 
 3   customer_city                99441 non-null  object
 4   Estado_cliente               99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


In [43]:
customers.isnull().sum()

id_cliente                     0
identificador_unico_cliente    0
codigo_postal_cliente          0
customer_city                  0
Estado_cliente                 0
dtype: int64

## 1.8 EDA para dataset: sellers.

In [44]:
sellers.head(2)

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP


In [18]:
sellers = sellers.rename(columns={
    'seller_id': 'id_vendedor',
    'seller_zip_code_prefix': 'codigo_postal_vendedor',
    'seller_city': 'ciudad_vendedor',
    'seller_state': 'stado vendedor'
})

In [46]:
sellers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   id_vendedor             3095 non-null   object
 1   codigo_postal_vendedor  3095 non-null   int64 
 2   ciudad_vendedor         3095 non-null   object
 3   stado vendedor          3095 non-null   object
dtypes: int64(1), object(3)
memory usage: 96.8+ KB


In [47]:
sellers.isnull().sum()

id_vendedor               0
codigo_postal_vendedor    0
ciudad_vendedor           0
stado vendedor            0
dtype: int64

# 2- Análisis con SQL

In [19]:
orders['Días_Diferencia'] = (orders['fecha_entregada_cliente'] - orders['fecha_estimada_entrega']).dt.days

In [21]:
condiciones = [
    (orders['Días_Diferencia'] < -5),
    (orders['Días_Diferencia'] <= 0),
    (orders['Días_Diferencia'] <= 5),
    (orders['Días_Diferencia'] > 5)
]

Nombre_rangos = [
    'Muy_Anticipado',
    'A_Tiempo',
    'Retraso_Leve',
    'Retraso_Critico'
]

orders['Rango_Entrega'] = np.select(condiciones, Nombre_rangos, default='Sin_dato')

orden_logico = [1,2,3,4]
orders['Rango_Entrega_Id'] = np.select(condiciones, orden_logico, default=5)

In [27]:
orders.head(2)

,id_orden,id_cliente,estado_orden,fecha_compra,Hora_orden_aprovada,fecha_entrega_transportista,fecha_entregada_cliente,fecha_estimada_entrega,Días_Diferencia,Rango_Entrega,Rango_Entrega_Id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,-8.0,Muy_Anticipado,1
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,-6.0,Muy_Anticipado,1


# 3- Exportación de datasets.

In [22]:
path_ = 'C:\\Users\\luis\\Dropbox\\Data-science\\Proyectos\\Analisis-Comercial\\Dataset2\\datasets_limpios\\'

In [23]:
orders.to_csv(path_+'clean_orders.csv', sep=',', header=True)
order_items.to_csv(path_+'clean_orders_items.csv', sep=',', header=True)
payments.to_csv(path_+'clean_payments.csv', sep=',', header=True)
reviews.to_csv(path_+'clean_reviews.csv', sep='|', header=True)
products.to_csv(path_+'clean_products.csv', sep=',', header=True)
customers.to_csv(path_+'clean_customers.csv', sep=',', header=True)
sellers.to_csv(path_+'clean_sellers.csv', sep=',', header=True)

In [25]:
import csv

In [26]:
reviews.to_csv(path_+'clean_reviews.csv', sep=';', index=False, encoding='utf-8-sig',
               quoting=csv.QUOTE_ALL, header=True)